<a href="https://colab.research.google.com/github/kuds/rl-mujoco-tennis/blob/main/notebooks/sb3_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Courtside Dynamics: SB3 training

One notebook for the whole curriculum. Pick an environment (`BallBalance`, `BallBounce`, `WallBall`, `TennisWall`) and an algorithm (`SAC` or `PPO`) at the top, then run all cells.

Each environment's defaults (training budget, custom CSV rows, phase labels for the state-machine reward) live in `courtside_dynamics.recipes`, so adding an env is one entry in that registry -- this notebook needs no edits.

## 1. Install

In [ ]:
!pip install -q "courtside-dynamics[train,notebooks] @ git+https://github.com/kuds/rl-mujoco-tennis"

## 2. Choose environment & algorithm

Set `ENV` and `ALGO` here -- everything below picks them up automatically.

* `USE_DRIVE = True` mounts Google Drive so checkpoints, eval npz, replay videos, and learning-curve plots survive a Colab runtime restart. Falls back to local logs if Drive isn't available.
* `QUICK_TEST = True` runs the whole pipeline (training, evaluation, video, plot) in a couple of minutes against a tiny budget. Use it to smoke-test a new runtime before committing to a real run.

In [ ]:
ENV = "TennisWall"   # BallBalance | BallBounce | WallBall | TennisWall
ALGO = "SAC"         # SAC | PPO
USE_DRIVE = False
QUICK_TEST = False

# Optional: override the recipe's default training budget.
# Leave as None to use the recipe default (1M-2M steps).
TOTAL_TIMESTEPS = None

## 3. Mount Google Drive (optional) and pick a run directory

Each call to `resolve_run_dir(ENV, ALGO)` creates a fresh, timestamped directory so re-runs don't clobber prior artifacts. Layout:

```
<root>/<env>/<algo>/<YYYYMMDD_HHMMSS>/
  best_model.zip          final_model.zip
  evaluations.npz         monitor/*.monitor.csv
  tensorboard/            videos/
  run_config.json         run_summary.txt
  learning_curve.png      best_model.mp4
```

Root is `MyDrive/Finding Theta/courtside-dynamics/training_runs/` when Drive is mounted, otherwise `./logs/`.

In [ ]:
from courtside_dynamics.notebook_utils import mount_drive, resolve_run_dir

if USE_DRIVE:
    mount_drive()

LOG_DIR = resolve_run_dir(ENV, ALGO, use_drive=USE_DRIVE)
print("Logging to:", LOG_DIR)

## 4. Configure Colab GPU

Sets up EGL so MuJoCo can render off-screen on the Colab GPU. No-op outside of Colab.

In [ ]:
from courtside_dynamics.colab_setup import setup_colab
setup_colab()

## 5. Build the training config

`build_train_config` looks up the recipe for `ENV`, fills in the per-env extras (custom CSV rows for Ball Bounce, phase labels for Tennis Wall, ...), and returns a `TrainConfig` ready for `train()`.

In [ ]:
from courtside_dynamics.recipes import RECIPES, build_train_config

print(f"Recipe: {ENV} -> {RECIPES[ENV].description}")

cfg = build_train_config(
    ENV,
    algo=ALGO,
    log_dir=LOG_DIR,
    total_timesteps=TOTAL_TIMESTEPS,
    quick_test=QUICK_TEST,
)
print(
    f"algo={cfg.algo}  total_timesteps={cfg.total_timesteps:,}  "
    f"eval_freq={cfg.eval_freq:,}"
)

## 6. Train

`train(cfg)` builds vectorized train + eval envs, attaches `EvalCallback`, `VideoRecordCallback`, and `InfoDictEvalCallback`, and runs SB3's `model.learn`. The best policy seen during evaluation is saved to `LOG_DIR/best_model.zip`.

In [ ]:
from courtside_dynamics.training import train

model = train(cfg)

## 7. Learning curves

Per-episode training rewards (left) come from `LOG_DIR/monitor/*.monitor.csv`. Deterministic eval rewards (right, mean +/- std) come from `LOG_DIR/evaluations.npz`.

In [ ]:
import os
from courtside_dynamics.notebook_utils import plot_learning_curve

plot_learning_curve(
    LOG_DIR,
    save_path=os.path.join(LOG_DIR, "learning_curve.png"),
)

## 8. Replay the best model

Loads `best_model.zip` from `LOG_DIR`, rolls it out deterministically, encodes the frames as MP4, and embeds the clip in this notebook.

In [ ]:
from courtside_dynamics.notebook_utils import (
    record_best_model_video,
    display_video,
)

video_path = record_best_model_video(
    LOG_DIR,
    cfg.env_fn,
    algo=ALGO,
    video_length=750,
)
display_video(video_path)

## 9. Disconnect Colab runtime (optional)

Frees the GPU when you're done so the next run can grab a fresh runtime. Uncomment to enable. No-op outside of Colab.

In [ ]:
from courtside_dynamics.notebook_utils import disconnect_runtime
# disconnect_runtime()